In [ ]:
import cv2
import os
from glob import glob
from tqdm import tqdm

# Mask extraction lives in mask_extraction.py so the notebooks, the training
# pipeline and test.py all share one implementation.
from mask_extraction import edit_mask, DELTA_E_THRESHOLD

print(f"Using CIE76 deltaE threshold {DELTA_E_THRESHOLD}")

In [5]:
original_images = glob(os.path.join("data", "images", "*.jpg"))
edited_images = glob(os.path.join("data", "results", "*.jpg"))

mappings = {}
for orig_path in original_images:
    base_name = os.path.basename(orig_path)
    edited_path = os.path.join("data", "results", base_name)
    # os.path.exists beats membership in the glob list: it does not depend on the
    # two paths being spelled with identical separators.
    if os.path.exists(edited_path):
        mappings[orig_path] = edited_path
original_images = list(mappings.keys())
edited_images = list(mappings.values())

In [ ]:
os.makedirs(os.path.join("data", "masks"), exist_ok=True)

for paths in tqdm(zip(original_images, edited_images), desc="Generating edit maps", total=len(original_images)):
    original_image = cv2.imread(paths[0])
    edited_image = cv2.imread(paths[1])
    if original_image is None or edited_image is None:
        print(f"Warning: Could not load image(s): {paths[0]}, {paths[1]}")
        continue
    if original_image.shape != edited_image.shape:
        print(f"Warning: Image shapes do not match: {paths[0]}, {paths[1]}")
        continue
    # LAB deltaE, not a grayscale difference -- grayscale is blind to edits that
    # change hue at constant luminance. See mask_extraction.py.
    mask = edit_mask(original_image, edited_image)
    cv2.imwrite(os.path.join("data", "masks", os.path.basename(paths[0])), mask)